## LLM 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성
- `2026.03.25` : logged_call_llm / update_log_parsed_output 테스트 추가, uncached 파라미터 테스트 강화

In [1]:
import sys

from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, "../../")

### 1. LLM 인스턴스 생성

In [2]:
from backend.app.agent.llm import get_llm, get_llm_uncached
from backend.app.utils.settings import get_settings

settings = get_settings()
print(f"LLM Provider: {settings.llm_provider}")
print(f"OpenAI Model: {settings.openai_model}")
print(f"Anthropic Model: {settings.anthropic_model}")

LLM Provider: openai
OpenAI Model: gpt-4o
Anthropic Model: claude-sonnet-4-5-20250929


In [4]:
# 기본 LLM 인스턴스 (캐시됨)
llm = get_llm()

In [5]:
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x113911790>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11420d100>, root_client=<openai.OpenAI object at 0x1139134d0>, root_async_client=<openai.AsyncOpenAI object at 0x1140ff350>, model_name='gpt-4o', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, max_tokens=4096)

In [6]:
type(llm)

langchain_openai.chat_models.base.ChatOpenAI

In [7]:
print(f"LLM Type: {type(llm).__name__}")
print(f"Model: {llm.model_name if hasattr(llm, 'model_name') else llm.model}")

LLM Type: ChatOpenAI
Model: gpt-4o


### 2. 기본 호출 테스트

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain 메시지 방식 직접 호출
messages = [
    SystemMessage(content="당신은 친절한 AI 어시스턴트입니다. 간결하게 답변하세요."),
    HumanMessage(content="B2B AI 개발 프로젝트에서 Go/No-Go 판단이 중요한 이유를 한 문장으로 설명하세요."),
]

response = await llm.ainvoke(messages)
print(f"Response type: {type(response).__name__}")
print(f"Content: {response.content}")

Response type: AIMessage
Content: B2B AI 개발 프로젝트에서 Go/No-Go 판단은 자원의 효율적 사용과 프로젝트 성공 가능성을 극대화하기 위해 필수적입니다.


### 3. call_llm 유틸리티 테스트

In [ ]:
from backend.app.agent.base import call_llm

result = await call_llm(
    system_prompt="당신은 B2B AI Deal 평가 전문가입니다. 한국어로 간결하게 답변하세요.",
    user_prompt="AI 개발 프로젝트의 수익성 평가 시 고려해야 할 핵심 요소 3가지를 나열하세요.",
)
print(result)

In [ ]:
# 특정 LLM 인스턴스를 주입하여 호출
custom_llm = get_llm_uncached(temperature=0.0, max_tokens=256)

result = await call_llm(
    system_prompt="당신은 B2B AI Deal 평가 전문가입니다.",
    user_prompt="'기술 적합성' 평가 기준을 한 문장으로 정의하세요.",
    llm=custom_llm,
)
print(result)

### 4. 파라미터 변경 테스트

In [ ]:
# temperature 비교: 0.0 (결정적) vs 0.9 (창의적)
prompt = "AI 프로젝트의 리스크를 한 단어로 표현하면?"

llm_deterministic = get_llm_uncached(temperature=0.0)
llm_creative = get_llm_uncached(temperature=0.9)

for label, model in [("temperature=0.0", llm_deterministic), ("temperature=0.9", llm_creative)]:
    result = await call_llm(
        system_prompt="한국어로 한 단어만 답변하세요.",
        user_prompt=prompt,
        llm=model,
    )
    print(f"[{label}] {result.strip()}")

In [ ]:
# max_tokens 제한 테스트
llm_short = get_llm_uncached(max_tokens=50)

result = await call_llm(
    system_prompt="한국어로 답변하세요.",
    user_prompt="AI 개발 프로젝트의 전체 생애주기를 상세히 설명해주세요.",
    llm=llm_short,
)
print(f"max_tokens=50 결과 (잘림 확인):\n{result}")

### 5. logged_call_llm 테스트 (DB 로깅 포함)

`logged_call_llm`은 LLM 호출 결과를 AgentLog 테이블에 자동 기록합니다.  
**사전 요구사항:** PostgreSQL 실행 (`make docker-up && make migrate && make seed`)

In [ ]:
import uuid

from backend.app.agent.base import logged_call_llm, update_log_parsed_output

# 테스트용 deal_id 생성
test_deal_id = uuid.uuid4()

# logged_call_llm 호출 — DB에 AgentLog 레코드 생성됨
raw_output, log_id = await logged_call_llm(
    system_prompt="당신은 B2B AI Deal 평가 전문가입니다. JSON으로 응답하세요.",
    user_prompt='다음 형식으로 응답: {"risk_level": "high/medium/low", "reason": "한 문장 설명"}\n프로젝트: 공장 설비 예지보전 AI',
    deal_id=test_deal_id,
    node_name="llm_test",
)

print(f"log_id: {log_id}")
print(f"raw_output:\n{raw_output}")

In [ ]:
# parsed_output 업데이트 — AgentLog 레코드에 파싱된 결과 저장
parsed = parse_json_response(raw_output)
await update_log_parsed_output(log_id, parsed)
print(f"parsed_output 업데이트 완료: {parsed}")

In [ ]:
# DB에서 로그 레코드 직접 조회하여 검증
from backend.app.db.repositories import agent_log_repo
from backend.app.db.session import AsyncSessionLocal

async with AsyncSessionLocal() as session:
    log = await agent_log_repo.get(session, log_id)
    if log:
        print(f"✓ AgentLog 레코드 확인")
        print(f"  deal_id: {log.deal_id}")
        print(f"  node_name: {log.node_name}")
        print(f"  duration_ms: {log.duration_ms}ms")
        print(f"  raw_output 길이: {len(log.raw_output or '')} chars")
        print(f"  parsed_output: {log.parsed_output}")
        print(f"  error: {log.error}")
    else:
        print(f"✗ log_id={log_id} 레코드를 찾을 수 없음")

### 6. JSON 응답 파싱 테스트

In [ ]:
from backend.app.agent.base import parse_json_response

# Case 1: 정상 JSON 문자열
raw_json = '{"score": 85, "verdict": "go"}'
parsed = parse_json_response(raw_json)
print(f"Case 1 (순수 JSON): {parsed}")

In [ ]:
# Case 2: Markdown 코드블록으로 감싼 JSON
raw_markdown = """분석 결과입니다:
```json
{"score": 72, "verdict": "go", "rationale": "기술 적합성이 높음"}
```"""
parsed = parse_json_response(raw_markdown)
print(f"Case 2 (코드블록 JSON): {parsed}")

In [ ]:
# Case 3: 파싱 실패 케이스 — 일반 텍스트
try:
    parse_json_response("이것은 JSON이 아닙니다.")
except ValueError as e:
    print(f"Case 3 (일반 텍스트): {e}")

# Case 3-2: 파싱 실패 케이스 — 빈 문자열
try:
    parse_json_response("")
except ValueError as e:
    print(f"Case 3-2 (빈 문자열): {e}")

In [ ]:
# Case 4: LLM에 JSON 출력 요청 → 파싱
raw_output = await call_llm(
    system_prompt="반드시 JSON 형식으로만 응답하세요.",
    user_prompt='다음 형식으로 응답하세요: {"project": "프로젝트명", "risk_level": "high/medium/low"}\n프로젝트: AI 챗봇 개발',
)
print(f"LLM 원본 응답:\n{raw_output}\n")

parsed = parse_json_response(raw_output)
print(f"파싱 결과: {parsed}")